# 01 · Dataset de entrenamiento y validación

Genera los pares pregunta–SQL sintéticos sobre 10 esquemas propios, los **valida en MySQL 8.4.10** (solo se conservan los que ejecutan sin error) y registra los descartes. También arma el set de validación propia (2 esquemas que no aparecen en entrenamiento, 40 preguntas).


In [ ]:
# Clonar el repositorio en Colab (ajusta la URL a tu repo privado)
# !git clone https://github.com/USUARIO/proyecto-dl-3.git
# %cd proyecto-dl-3
import sys, os
# Si ejecutas localmente, asegúrate de estar en la raíz del repo.
assert os.path.exists("src"), "Ejecuta desde la raíz del repositorio."


## 1. Levantar MySQL 8.4.10
En Colab se instala el servidor; en local puedes usar Docker:
`docker run -d -e MYSQL_ROOT_PASSWORD=root -p 3306:3306 mysql:8.4.10`

In [ ]:
# --- Colab: instalar MySQL 8.4 y pymysql ---
# !pip -q install pymysql
# (En Colab conviene usar la imagen de Docker o el tar oficial 8.4.10.)
# Aquí asumimos un MySQL 8.4.10 accesible en 127.0.0.1:3306 (root/root).
import pymysql  # noqa


## 2. Generar y exportar los pares sintéticos

In [ ]:
from src.exportar_datos import exportar_entrenamiento, exportar_validacion_propia
pares = exportar_entrenamiento()
val = exportar_validacion_propia()


## 3. Validar en MySQL y filtrar descartes
Reescribe `train.jsonl` solo con los pares que ejecutan sin error y guarda `datos/descartes/descartes.csv` con el motivo de cada descarte.

In [ ]:
import json
from src.generar_datos import generar_entrenamiento
from src.validar_mysql import validar_pares, categoria_error
from collections import Counter
import csv, os

pares = generar_entrenamiento()
ok, desc = validar_pares(pares, host='127.0.0.1', port=3306, user='root', password='root')
print('Conservados:', len(ok), '| Descartados:', len(desc))
print('Descartes por categoría:', dict(Counter(categoria_error(d['motivo_descarte']) for d in desc)))

# Reescribir train.jsonl solo con los validados (deduplicados)
vistos=set(); 
with open('datos/entrenamiento/train.jsonl','w',encoding='utf-8') as f:
    for p in ok:
        k=(p['esquema'],p['pregunta'].strip().lower(),p['sql'].strip())
        if k in vistos: continue
        vistos.add(k)
        f.write(json.dumps({'esquema':p['ddl'],'esquema_nombre':p['esquema'],'pregunta':p['pregunta'],'sql':p['sql'],'nivel':p['nivel'],'peso':p['peso']},ensure_ascii=False)+'\n')

os.makedirs('datos/descartes',exist_ok=True)
with open('datos/descartes/descartes.csv','w',newline='',encoding='utf-8') as f:
    w=csv.writer(f); w.writerow(['esquema','nivel','pregunta','sql','categoria','motivo'])
    for d in desc: w.writerow([d['esquema'],d['nivel'],d['pregunta'],d['sql'],categoria_error(d['motivo_descarte']),d['motivo_descarte']])
print('Listo. Pares finales:', len(vistos))


## 4. Resumen del dataset

In [ ]:
import json
n=sum(1 for _ in open('datos/entrenamiento/train.jsonl'))
from collections import Counter
niv=Counter(json.loads(l)['nivel'] for l in open('datos/entrenamiento/train.jsonl'))
print('Total pares:', n)
print('Por nivel:', dict(niv))
